# Ashiana Similar Products — design notebook

A content-based, cold-start "similar products" recommender for Ashiana, a handcrafted
ethnic jewelry brand — built from frozen pretrained encoders, hand-designed metadata
similarity, and weighted late fusion (no training, no click data).

This notebook covers the design reasoning, signal analysis, and evaluation behind the
project. It's read-only against the committed pipeline outputs: it loads the same
artifact bundle the API and frontend serve, and it does not re-run any embedding model
or (by default) call the LLM. **"Run all" works on a fresh Colab runtime with no
secrets** — see the Setup section below.

Repo: `<FILL IN ONCE THE GITHUB REPO IS PUBLIC — see docs/DECISIONS.md>`


## Setup

Clones the repo if it isn't already checked out (skipped when this notebook is run
from inside a local clone), installs the pinned offline-pipeline dependencies, and
imports `core/` and `pipeline/` directly — the same modules the API and the batch build
script use, so nothing in this notebook reimplements ranking logic (rule 4).


In [ ]:
import os
import subprocess
import sys

REPO_URL = "<FILL IN ONCE THE GITHUB REPO IS PUBLIC>"
REPO_DIR = "ashiana-similar-products"

if os.path.basename(os.getcwd()) != REPO_DIR and not os.path.exists(os.path.join(REPO_DIR, "core")):
    if "FILL IN" in REPO_URL:
        raise RuntimeError(
            "REPO_URL is still a placeholder and this doesn't look like a checkout of "
            "the repo already. Set REPO_URL above, or run this notebook from inside "
            "a clone of the repo."
        )
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

if os.path.basename(os.getcwd()) != REPO_DIR and os.path.exists(os.path.join(REPO_DIR, "core")):
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())


In [ ]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q -r requirements-pipeline.txt
else:
    # Assume a local dev checkout already has requirements-pipeline.txt installed
    # (e.g. via `make setup`) — reinstalling on every local run is unnecessary.
    print("Not running in Colab — skipping pip install (assumes `make setup` already ran).")


In [ ]:
import io
import json
import random
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import requests
from IPython.display import HTML, display
from PIL import Image

from core.bundle import load_bundle
from pipeline.paths import ARTIFACTS_DIR, ROOT
from pipeline.recommend import recommend

plt.rcParams["figure.facecolor"] = "white"

bundle = load_bundle(ARTIFACTS_DIR)
print(
    f"Loaded bundle {bundle.manifest['bundle_version']} "
    f"\u2014 {bundle.manifest['n_items']} products"
)


### Display helpers

Product photography (`data/images/`, `artifacts/thumbs/`) is gitignored per D3 —
image rights aren't cleared for a public repo yet (`docs/DECISIONS.md`), so nothing
image-derived is committed and none of it exists on a fresh clone. Every image shown
below is instead fetched live, at notebook-run time, from the product's own source URL
(`image_urls` in the committed catalog — a factual data field, not image bytes) and
never written back to the repo. If a source host is unreachable, `fetch_image` falls
back to a plain placeholder rather than raising, so a flaky external host can never
break "Run all."


In [ ]:
IMG_TIMEOUT_S = 8
_HEADERS = {"User-Agent": "Mozilla/5.0 (ashiana-similar-products notebook)"}
_image_cache: dict[str, Image.Image] = {}


def fetch_image(sku: str, size: int = 220) -> Image.Image:
    if sku in _image_cache:
        return _image_cache[sku]
    urls = bundle.catalog.get(sku, {}).get("image_urls") or []
    img = None
    if urls:
        try:
            resp = requests.get(urls[0], timeout=IMG_TIMEOUT_S, headers=_HEADERS)
            resp.raise_for_status()
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
        except Exception:
            img = None
    if img is None:
        img = Image.new("RGB", (size, size), (230, 226, 220))
    img.thumbnail((size, size))
    _image_cache[sku] = img
    return img


def show_row(
    skus: list[str], labels: dict[str, str] | None = None, figsize_per: float = 2.2
) -> None:
    n = len(skus)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per * n, figsize_per + 0.7))
    axes = [axes] if n == 1 else axes
    for ax, sku in zip(axes, skus, strict=True):
        ax.imshow(fetch_image(sku))
        ax.axis("off")
        label = labels.get(sku) if labels else bundle.catalog[sku]["title"]
        ax.set_title(label[:30] + ("\u2026" if len(label) > 30 else ""), fontsize=8)
    plt.tight_layout()
    plt.show()


def show_table(headers: list[str], rows: list[list]) -> None:
    th_style = "text-align:left;padding:4px 10px;border-bottom:2px solid #4A2712"
    td_style = "padding:4px 10px;border-bottom:1px solid #ddd"
    thead = "".join(f"<th style='{th_style}'>{h}</th>" for h in headers)
    body = "".join(
        "<tr>" + "".join(f"<td style='{td_style}'>{c}</td>" for c in row) + "</tr>"
        for row in rows
    )
    table_style = "border-collapse:collapse;font-size:13px"
    display(HTML(f"<table style='{table_style}'>{thead}{body}</table>"))


## 1. Problem, constraints, and what "similar" means here

**Problem.** Ashiana has no user behavior data — no clicks, purchases, carts, or ratings.
A shopper landing on a product page today needs a "similar pieces" rail, but every
standard recommender approach (collaborative filtering, learned rankers, session-based
models) needs interaction history this catalog doesn't have. This is a pure **cold-start**
setting: everything has to come from the content itself — photos, titles, descriptions,
and structured metadata.

**What "similar" means (LOCKED, §3).** Substitutes only: other pieces a shopper might buy
*instead of* this one. Not complements ("complete the look" — a matching necklace for a
pair of earrings is explicitly out of scope), not accessories-for-an-outfit, not
"frequently bought together." A same-product-type hard filter enforces this directly —
a ring's "similar pieces" are other rings, never a top suggested pendant.

**Constraints.**
- Free-tier hosting only (the API sleeps when idle; §10.3's cold-start banner covers it).
- No training or fine-tuning — frozen pretrained encoders (DINOv2-small for images,
  bge-small-en-v1.5 for text) plus hand-designed metadata similarity, combined by a
  weighted sum (§3 LOCKED). No model runs at serve time; every embedding and similarity
  matrix is precomputed offline.
- Roughly 10 working sessions end-to-end, across data collection, the image/text/metadata
  pipelines, evaluation, the API, the frontend, and this notebook.
- Catalog size: the roadmap estimated 30–100 SKUs; the real scraped catalog turned out
  to be **472 products across 7 product types** (see §2 below) — large enough that a few
  product types (brooch, home_decor, bangle_bracelet, ring — 9–10 items each) are thin
  enough to matter for the category-fallback rule (§3 LOCKED) and for evaluation variance.

**Approach at a glance.** Three independent similarity signals per product pair — image
cosine, text cosine, metadata (Gower-style) similarity — each z-scored over its
off-diagonal entries (§4 below explains why), then combined as
`score = w_image·Z_image + w_text·Z_text + w_meta·Z_meta`, filtered to the same product
type (with a documented fallback when too few same-type candidates exist), and ranked.


## 2. Data overview

Counts, price distribution, and missingness are all pulled from the committed data
validation report (`artifacts/eval/data_report.json`, written once by
`pipeline/clean.py` when the catalog was built) — reused here, not recomputed, so the
numbers in this notebook always match what the pipeline itself validated against.


In [ ]:
data_report = json.loads((ARTIFACTS_DIR / "eval" / "data_report.json").read_text())
print(f"{data_report['n_products']} products \u00b7 snapshot {data_report['snapshot_date']}")

show_table(
    ["product type", "count"],
    sorted(data_report["counts_by_product_type"].items(), key=lambda kv: -kv[1]),
)


In [ ]:
show_table(
    ["collection", "count"],
    sorted(data_report["counts_by_collection"].items(), key=lambda kv: -kv[1]),
)
print(
    "Most products carry no collection tag at all (\"(none)\" above) \u2014 metadata "
    "similarity's Gower-style renormalization (\u00a78 Phase 3) is what makes that workable: "
    "a missing field drops out of the average instead of forcing a guess."
)


In [ ]:
show_table(
    ["field", "missing share"],
    [[field, f"{share:.1%}"] for field, share in data_report["missingness"].items()],
)


In [ ]:
prices = sorted(p["price_inr"] for p in bundle.catalog.values() if p.get("price_inr") is not None)
plt.figure(figsize=(6.5, 3))
plt.hist(prices, bins=30, color="#4A2712")
plt.xlabel("price (\u20b9)")
plt.ylabel("number of products")
plt.title(f"Price distribution \u2014 n={len(prices)}, median \u20b9{prices[len(prices)//2]:.0f}, "
          f"range \u20b9{prices[0]:.0f}\u2013\u20b9{prices[-1]:.0f}")
plt.tight_layout()
plt.show()


Sample packshots (fetched live — see the note on image availability above):

In [ ]:
random.seed(0)
sample_skus = random.sample(list(bundle.catalog.keys()), 6)
show_row(sample_skus, figsize_per=2.3)


## 3. Preprocessing

### Pad vs. crop

Packshots are padded to a square with white, not center-cropped, before resizing to
224×224 for DINOv2 (§3 LOCKED) — a center crop would cut off dangling earrings and
necklace ends, which this catalog has a lot of. `_pad_to_square` below is the *actual*
`pipeline/images.py` function, not a reimplementation; the center-crop panel is a
one-off comparison written only for this cell; it's never called by the real pipeline.


In [ ]:
from pipeline.images import PAD_COLOR, _pad_to_square

# Padding vs. cropping only look different on a non-square photo, so search a
# few candidates for one with a clearly non-square aspect ratio rather than
# risking an already-square pick that would make all three panels identical.
random.seed(1)
pad_demo_candidates = random.sample(list(bundle.catalog.keys()), 15)

original = None
for candidate_sku in pad_demo_candidates:
    url = (bundle.catalog[candidate_sku].get("image_urls") or [None])[0]
    if not url:
        continue
    try:
        resp = requests.get(url, timeout=IMG_TIMEOUT_S, headers=_HEADERS)
        resp.raise_for_status()
        candidate_img = Image.open(io.BytesIO(resp.content)).convert("RGB")
    except Exception:
        continue
    w, h = candidate_img.size
    if max(w, h) / min(w, h) >= 1.2:
        pad_demo_sku, original = candidate_sku, candidate_img
        break
else:
    print(
        "No sufficiently non-square candidate reachable in this sample; "
        "skipping the pad-vs-crop panel."
    )

if original is not None:
    padded = _pad_to_square(original, PAD_COLOR)

    w, h = original.size
    side = min(w, h)
    left, top = (w - side) // 2, (h - side) // 2
    cropped = original.crop((left, top, left + side, top + side))  # NOT used by the pipeline

    fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.6))
    for ax, img, label in zip(
        axes,
        [original, padded, cropped],
        [
            "original",
            "padded (what the pipeline uses)",
            "center-cropped (not used \u2014 comparison only)",
        ],
        strict=True,
    ):
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(label, fontsize=9)
    plt.tight_layout()
    plt.show()


### Removed boilerplate

In [ ]:
boilerplate = json.loads((ARTIFACTS_DIR / "eval" / "removed_boilerplate.json").read_text())
_min_share = boilerplate["boilerplate_min_share"]
print(
    f"A sentence counts as boilerplate once it appears in \u2265{_min_share:.0%} "
    f"of products' descriptions ({boilerplate['n_products']} products checked)."
)
show_table(
    ["template sentence", "example", "# products", "share"],
    [
        [r["template"], r["example"], r["n_products"], f"{r['share']:.1%}"]
        for r in boilerplate["removed_sentences"]
    ],
)


### LLM descriptor: before / after

In [ ]:
before_after = json.loads((ARTIFACTS_DIR / "eval" / "descriptor_before_after.json").read_text())
show_table(
    ["title", "raw description (truncated)", "descriptor sentence"],
    [
        [r["title"][:45], (r["description_raw"] or "")[:70] + "\u2026", r["descriptor_sentence"]]
        for r in before_after[:6]
    ],
)
print(f"({len(before_after)} products in the full before/after table)")


### Grounding log

Every term the LLM extracts into `materials`/`stones`/`colors` must actually appear in
the input text (after synonym mapping) — ungrounded terms are dropped and logged, never
silently kept (§8 Phase 2).


In [ ]:
grounding_log = json.loads((ARTIFACTS_DIR / "eval" / "grounding_log.json").read_text())
n_with_drops = sum(1 for v in grounding_log.values() if any(v["dropped_extracted"].values()))
n_with_flags = sum(1 for v in grounding_log.values() if v.get("flagged_ungrounded_not_dropped"))
print(
    f"{len(grounding_log)} products logged \u2014 {n_with_drops} had an ungrounded "
    f"extracted term dropped, {n_with_flags} had an ungrounded non-extracted term "
    "flagged (kept, for review)."
)

example_sku, example = next(iter(grounding_log.items()))
print(f"\nExample ({bundle.catalog[example_sku]['title'][:50]}):")
print(json.dumps(example, indent=2))


### Optional: live LLM rerun

The cell below only runs if `LLM_API_KEY` is actually available (checked in Colab
secrets first, then the environment) — **every other cell in this notebook uses the
committed `artifacts/descriptors.json` cache and never needs this.** If a key is found,
it re-runs `pipeline.describe.run_llm_descriptors` for real. Verified by actually
running it with a real key: 467/472 products are cache hits
(`sha256(input + prompt_version + model)`, zero API calls); the remaining ~5 are
products that previously fell back to a taxonomy-stripped title
(`extraction_failures.json`) and so never had a valid cached extraction to hit — those
retry every run by design, not a cache bug. Running this cell rewrites
`data/catalog.jsonl`, `artifacts/eval/grounding_log.json`, and
`artifacts/eval/extraction_failures.json` on disk (the same files
`pipeline.describe.main()` writes) — harmless on a throwaway Colab runtime, worth
knowing if running this notebook against a real local checkout.


In [ ]:
def _get_llm_api_key() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        try:
            return userdata.get("LLM_API_KEY")
        except Exception:
            return None
    except ImportError:
        return os.environ.get("LLM_API_KEY")


_llm_api_key = _get_llm_api_key()

if _llm_api_key:
    os.environ["LLM_API_KEY"] = _llm_api_key
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
    os.environ.setdefault("LLM_MODEL", "openai/gpt-oss-20b")
    print(
        "LLM_API_KEY found \u2014 re-running descriptor extraction "
        "(mostly cache hits expected, see docs/DECISIONS.md)\u2026"
    )

    from pipeline.describe import _load_catalog as _describe_load_catalog
    from pipeline.describe import run_llm_descriptors, strip_boilerplate

    _products = _describe_load_catalog()
    _cleaned, _ = strip_boilerplate(_products)
    run_llm_descriptors(_products, _cleaned)
    print(
        "Done \u2014 see the call count above. A handful of calls (previously-failed "
        "products retrying) is expected; hundreds would mean the cache didn't match."
    )
else:
    print(
        "No LLM_API_KEY found (checked Colab secrets and the environment) \u2014 "
        "skipping the live rerun. This is expected and fine: this notebook's other "
        "cells all use the already-committed LLM cache."
    )


## 4. Signals

### Neighbor grids per signal

The same query product's nearest neighbors look different depending on which signal
alone is asked — image similarity favors visual style (motif, stone color, silhouette),
text similarity favors described design attributes, metadata similarity favors shared
collection/material/price. This is the raw `S_image`/`S_text`/`S_meta` matrices from the
committed bundle — no fusion yet.


In [ ]:
signal_query_sku = sample_skus[1]
q_idx = bundle.index_of(signal_query_sku)
print(f"Query: {bundle.catalog[signal_query_sku]['title']}")
show_row([signal_query_sku], figsize_per=2.4)

for signal in ("image", "text", "meta"):
    order = np.argsort(-bundle.S[signal][q_idx])
    neighbor_skus = [bundle.ids[i] for i in order if bundle.ids[i] != signal_query_sku][:5]
    print(f"\nTop-5 by {signal} alone:")
    show_row(neighbor_skus)


### Why z-score? Raw cosine per signal

Each signal's raw off-diagonal cosine similarity has a different center and spread —
combining raw cosines directly would let whichever signal happens to run "hotter"
dominate the fused score regardless of its actual weight. `core.fusion.offdiag_zscore`
puts every signal on the same scale (mean 0, std 1 over the off-diagonal) before the
weighted sum, so a weight like `w_image=0.6` means what it says.


In [ ]:
n = len(bundle.ids)
offdiag = ~np.eye(n, dtype=bool)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, signal in zip(axes, ("image", "text", "meta"), strict=True):
    vals = bundle.S[signal][offdiag]
    ax.hist(vals, bins=40, color="#4A2712")
    ax.set_title(f"{signal}\nmean={vals.mean():.3f}  std={vals.std():.3f}", fontsize=10)
    ax.set_xlabel("raw cosine similarity")
axes[0].set_ylabel("count (off-diagonal pairs)")
plt.suptitle(
    "Raw cosine distributions differ per signal \u2014 z-scoring is what "
    "makes the weights comparable",
    y=1.05,
)
plt.tight_layout()
plt.show()


## 5. Fusion

### Example recommendations

`pipeline.recommend.recommend` is the exact function the Phase 3 recommendations grid
and (in spirit — the API reimplements the same call against `core/` directly, §5
Phase 5) the live API both use. Default weights come from the bundle's tuned
`default_weights` (Phase 4's weight search).


In [ ]:
fusion_query_sku = signal_query_sku
result = recommend(bundle, fusion_query_sku, k=6)

query_price = bundle.catalog[fusion_query_sku].get("price_inr")
print(
    f"Query: {result['query_sku']} \u2014 {bundle.catalog[fusion_query_sku]['title']} "
    f"(\u20b9{query_price})"
)
print(f"Weights used: {result['weights_used']}")
show_row([fusion_query_sku], figsize_per=2.4)

rec_skus = [item["sku"] for item in result["items"]]
rec_labels = {
    item["sku"]: f"{item['title'][:24]}\u2026 ({item['score']:.2f})" for item in result["items"]
}
show_row(rec_skus, labels=rec_labels)

for item in result["items"]:
    tag = "  [fallback: related category]" if item["fallback"] else ""
    reasons_str = ", ".join(item["reasons"])
    print(f"  score={item['score']:6.3f}  {item['title'][:55]:55s}  {reasons_str}{tag}")


### Weight sensitivity

How the same query's top-5 list reorders as the three weights move — this is exactly
what the product page's inspector sliders let a reviewer do live (§10.2). Each column
here comes from a real `recommend()` call with different weights, not a simulation.


In [ ]:
WEIGHT_SETTINGS = {
    "image-heavy (0.8/0.1/0.1)": {"image": 0.8, "text": 0.1, "meta": 0.1},
    "text-heavy (0.1/0.8/0.1)": {"image": 0.1, "text": 0.8, "meta": 0.1},
    "meta-heavy (0.1/0.1/0.8)": {"image": 0.1, "text": 0.1, "meta": 0.8},
    "tuned default": None,
}

rows = []
for label, weights in WEIGHT_SETTINGS.items():
    r = recommend(bundle, fusion_query_sku, k=5, weights=weights)
    rows.append([label] + [f"{i + 1}. {item['title'][:32]}" for i, item in enumerate(r["items"])])

show_table(["weight setting", "#1", "#2", "#3", "#4", "#5"], rows)


## 6. Evaluation

Every number in this section is loaded straight from the committed
`artifacts/eval/report.json` and the committed figures under `artifacts/eval/figures/`
— written once by `pipeline/evaluate.py` (§8 Phase 4) and reused here, not recomputed.
Full methodology (pooling-bias caveat, the flat-region weight-selection procedure, proxy
scope) is in `report["methodology_notes"]`, printed at the end of this section.


In [ ]:
report = json.loads((ARTIFACTS_DIR / "eval" / "report.json").read_text())
labels = json.loads((ROOT / "labels" / "relevance_labels.json").read_text())["labels"]

label_counts = Counter(entry["label"] for entry in labels)
print(
    f"{report['n_queries']} stratified queries, {len(labels)} hand-labeled "
    f"query/candidate pairs, k={report['k']}"
)
show_table(
    ["relevance label", "meaning", "count"],
    [
        [0, "wrong / irrelevant", label_counts.get(0, 0)],
        [1, "acceptable", label_counts.get(1, 0)],
        [2, "I'd happily show this", label_counts.get(2, 0)],
    ],
)


In [ ]:
def _ci_str(metric: dict) -> str:
    return f"{metric['mean']:.3f} [{metric['ci_low']:.3f}, {metric['ci_high']:.3f}]"


rows = [
    [method, _ci_str(m["ndcg_at_5"]), _ci_str(m["p_at_5"])]
    for method, m in report["methods"].items()
]
show_table(["method", "NDCG@5 [95% CI]", "P@5 [95% CI]"], rows)
print(report["fused_vs_baseline"]["note"])


In [ ]:
wt = report["weight_tuning"]
print(f"Grid search: step {wt['grid_step']}, {wt['n_grid_points']} points.")
print(f"Chosen weights (flat-region selection, not the single best point): {wt['chosen_weights']} "
      f"\u2014 plateau size {wt['plateau_size']}/{wt['n_grid_points']}")
print(f"Mean NDCG@5 at chosen weights: {wt['chosen_weights_mean_ndcg_at_5']:.4f}")
loo = wt["leave_one_query_out_ndcg_at_5"]
print(f"Leave-one-query-out NDCG@5: {loo['mean']:.4f} [{loo['ci_low']:.4f}, {loo['ci_high']:.4f}] "
      "\u2014 close to the in-sample number, so the tuning doesn't look overfit to the 25 queries.")


In [ ]:
p = report["proxies"]
coverage = p["coverage"]
hubness = p["hubness"]
aug = p["image_augmentation_robustness"]
coverage_str = (
    f"{coverage['share']:.3f}  "
    f"({coverage['n_unique_recommended']}/{coverage['n_catalog']} products ever recommended)"
)
show_table(
    ["proxy", "value"],
    [
        [
            "held-out attribute agreement (meta-zeroed, vs. query's own collection)",
            f"{p['held_out_attribute_agreement']:.3f}",
        ],
        [
            "cross-signal agreement (Jaccard@5, image-only vs. text-only)",
            f"{p['cross_signal_agreement_jaccard5']:.3f}",
        ],
        ["coverage", coverage_str],
        ["hubness: mean in-degree", f"{hubness['mean_indegree']:.2f}"],
        ["hubness: in-degree skewness", f"{hubness['skewness']:.2f}"],
        [
            "diversity (mean intra-list similarity)",
            f"{p['diversity_mean_intra_list_similarity']:.3f}",
        ],
        ["price sanity (median query/rec price ratio)", f"{p['price_sanity_median_ratio']:.3f}"],
        ["image augmentation robustness", f"{aug['hits']}/{aug['n_samples']}"],
    ],
)

print("\nTop hubs (products that show up most often in others' top-5 lists):")
show_table(
    ["title", "in-degree"],
    [[h["title"][:55], h["count"]] for h in p["hubness"]["top_hubs"]],
)


In [ ]:
for fname in ("ndcg_by_method.png", "hubness_histogram.png", "weight_grid.png"):
    path = ARTIFACTS_DIR / "eval" / "figures" / fname
    display(Image.open(path))


In [ ]:
for note in report["methodology_notes"]:
    print(f"\u2022 {note}\n")


## 7. Failure cases and limitations

The five lowest-NDCG@5 queries under the tuned model, straight from
`report["failure_cases"]` — real evaluation output, not cherry-picked examples.


In [ ]:
for case in report["failure_cases"]:
    print(f"\nQuery: {case['query_title']}  (NDCG@5 = {case['ndcg_at_5']:.3f})")
    show_row([case["query_sku"]], figsize_per=1.8)
    for item in case["top5"]:
        print(f"    label={item['label']}  {item['title'][:60]}")


In [ ]:
from IPython.display import Markdown

_ft = report["methods"]["fused_tuned"]["ndcg_at_5"]
_io = report["methods"]["image_only"]["ndcg_at_5"]["mean"]
_materials_missing = data_report["missingness"]["materials"]
_collection_missing = data_report["missingness"]["collection"]

display(
    Markdown(
        f'''**Honest limitations, in the terms the evaluation actually measured**
(§8 Phase 4, `docs/DECISIONS.md`):

- **Only 25 labeled queries.** Every NDCG@5/P@5 number above carries a wide 95%
  bootstrap CI (e.g. `fused_tuned`: {_ft["ci_low"]:.2f}–{_ft["ci_high"]:.2f}). The results
  support "the fused model beats both baselines and image is doing most of the work"
  as a directional claim, not fine-grained rankings between methods whose CIs overlap.
- **Pooled-IR labeling bias.** Only candidates that appeared in one of six methods' top-8
  during label-pool construction were ever hand-labeled; anything else a method
  surfaces scores as relevance-0 by convention (standard TREC-style pooling), not as a
  confirmed miss. `fused_tuned` and most of `random_within_type`'s seeds can surface
  candidates outside that pool, so their scores are conservative lower bounds.
- **Thin categories.** Brooches, home decor, bangle/bracelets, and rings each have only
  9–10 products — barely enough same-type candidates to avoid the category-fallback
  rule, and the failure cases above skew toward exactly these thin categories (see the
  brooch and hair-accessory queries).
- **Image signal dominates, text/metadata add less than they might on a richer
  catalog.** `image_only` alone (NDCG@5 {_io:.3f}) is nearly indistinguishable from the
  fully tuned fusion ({_ft["mean"]:.3f}) — on a catalog this visually consistent
  (studio packshots, similar framing), DINOv2's embedding is carrying most of the
  substitutability signal. A catalog with richer free-text descriptions or more
  complete structured metadata (recall: {_materials_missing:.0%} of products are
  missing `materials`, {_collection_missing:.0%} missing `collection`) would likely
  shift this balance.
- **Single image per product.** Every scraped product had exactly one photo (no
  multi-angle shots), so there was never an actual packshot-selection *choice* to make
  — the border-whiteness heuristic in `pipeline/images.py` is exercised, but only in
  the "is this one photo usable" sense, not "which of several is best."
'''))


## 8. Scaling to 10k+ items

Everything above runs on N×N matrices (472×472 today) computed and stored in full —
fine at this scale, but it wouldn't stay fine forever. None of this is implemented; it's
the honest answer to "what would break first, and what would you do about it."

- **Store top-k lists instead of N×N matrices.** The API only ever reads row `q` of a
  fused score matrix and takes the top-k — it never needs the full N×N. Past a few
  thousand items, precomputing and storing each item's top-`k_max` neighbors *per
  signal* (not the full matrix) turns an O(N²) storage and lookup problem into O(N·k),
  at the cost of losing the ability to fuse with arbitrary weights at request time
  unless enough candidates are cached per signal to re-rank from.
- **Weighted cosine is a dot product over concatenated √w-scaled vectors.** Because
  `score = w_image·cos(a,b) + w_text·cos(a,b) + w_meta·cos(a,b)` and each `cos` is
  already a dot product of unit vectors, concatenating `[√w_image·e_image,
  √w_text·e_text, √w_meta·e_meta]` per item and taking a plain dot product of the
  concatenated vectors gives the identical fused score. That's what makes an ANN index
  usable at all here — ANN libraries index vectors under a single distance metric, not
  a runtime-adjustable weighted sum of three separate similarities.
- **ANN only around 100k+ items** — below that, exact search over precomputed matrices
  (today's approach) is fast enough and exactly correct, which an approximate index
  isn't. Past that point, the concatenated-vector trick above lets an ANN index (e.g.
  HNSW) serve the *default*-weight case, but the **filtered-search problem** bites
  immediately: the category hard-filter (§3 LOCKED) means a naive ANN query over the
  whole catalog can return a top-k with too few same-type results, needing either a
  per-product-type index or filtered/hybrid ANN search, and re-weighting away from the
  default breaks the precomputed index's distance metric entirely.
- **Incremental indexing and stale neighbor lists.** New products need embeddings
  computed and inserted without a full rebuild; ANN indexes (HNSW in particular) support
  incremental inserts but degrade over many of them without periodic reindexing, so
  "stale until the next rebuild" becomes an explicit tradeoff to manage rather than
  something a nightly `make build` sidesteps for free.
- **Hybrid ranking once click data exists.** The entire premise of this project is that
  there isn't any yet (§1). Once there is, the honest next step isn't to throw away the
  content-based signals — it's to blend them with collaborative or learned-ranker
  signals (e.g. as additional features in a re-ranker, or as a fallback for genuinely
  new items those signals can't yet cover), keeping content-based fusion as the
  cold-start floor rather than replacing it outright.


## 9. How to reproduce

```bash
git clone <FILL IN ONCE THE GITHUB REPO IS PUBLIC>
cd ashiana-similar-products
make setup      # venv + pinned requirements-pipeline.txt + playwright
make build      # runs the offline pipeline end-to-end from the committed
                 # data/catalog.jsonl and LLM cache -> artifacts/
make eval       # pipeline/evaluate.py -> artifacts/eval/report.json + figures
make api        # uvicorn api.app.main:app, http://localhost:8000
make ui         # cd frontend && npm run dev, http://localhost:5173
```

- `make build` makes **zero LLM calls** on a machine with no `LLM_API_KEY` set —
  every descriptor is already in `artifacts/descriptors.json`
  (`sha256(input + prompt_version + model)` → cached output), which is committed.
- This notebook itself only needs `pip install -r requirements-pipeline.txt` (handled
  above) plus network access to fetch product photos live for display — no API key,
  no GPU, no local image files.
- The full repo layout, canonical data schema, and every non-trivial decision made
  along the way (including the ones referenced throughout this notebook) are in
  `docs/DECISIONS.md`.
